# Bias-Adjusted Trimmed Mean PCE

This is a compact replication of the trimmed-mean PCE bias adjustment from the Cleveland Fed note. It uses:

- `outputs/trimmed_mean_rates.csv` for `trimmed_mean_12m`
- `outputs/kelly_skewness_minimal.csv` for `kelly_skewness_12m_avg`
- `cache/bea_NIUnderlyingDetail_U20404_M_1969_2026.csv` for headline PCE

The regression end date is computed from the latest available data date minus 18 months, because the dependent variable is a 36-month centered trimmed average of the trimmed-mean-minus-headline gap.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    display
except NameError:
    def display(obj):
        print(obj)


ROOT = Path.cwd()
TRIMMED_CSV = ROOT / "outputs" / "trimmed_mean_rates.csv"
SKEW_CSV = ROOT / "outputs" / "kelly_skewness_minimal.csv"
PCE_INDEX_CSV = ROOT / "cache" / "bea_NIUnderlyingDetail_U20404_M_1969_2026.csv"
OUTPUT_CSV = ROOT / "outputs" / "bias_adjusted_trimmed_mean_pce.csv"

REGRESSION_START = pd.Timestamp("1986-06-01")
FUTURE_MONTHS_NEEDED = 18
HAC_LAGS = 3


def centered_trimmed_average(series, before=17, after=18):
    x = np.asarray(series, dtype=float)
    out = np.full(len(x), np.nan)
    window_length = before + 1 + after
    for i in range(before, len(x) - after):
        window = x[i - before : i + after + 1]
        if len(window) == window_length and np.isfinite(window).all():
            out[i] = (window.sum() - window.min() - window.max()) / (window_length - 2)
    return out


def ols_hac(y, x, lags=3):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    xtx_inv = np.linalg.inv(X.T @ X)
    b = xtx_inv @ (X.T @ y)
    u = y - X @ b
    n, k = X.shape

    S = np.zeros((k, k))
    for t in range(n):
        xt = X[t : t + 1].T
        S += u[t] ** 2 * (xt @ xt.T)
    for L in range(1, lags + 1):
        w = 1 - L / (lags + 1)
        for t in range(L, n):
            xt = X[t : t + 1].T
            xl = X[t - L : t - L + 1].T
            S += w * u[t] * u[t - L] * (xt @ xl.T + xl @ xt.T)

    V = xtx_inv @ S @ xtx_inv * n / (n - k)
    se = np.sqrt(np.diag(V))
    tstat = b / se
    r2 = 1 - (u @ u) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, tstat, r2, n


trimmed = pd.read_csv(TRIMMED_CSV, parse_dates=["date"])
skew = pd.read_csv(SKEW_CSV, parse_dates=["date"])
pce = pd.read_csv(PCE_INDEX_CSV, parse_dates=["date"])

headline = pce.loc[pce["line"].eq(1), ["date", "value"]].sort_values("date")
headline["headline_pce_12m"] = 100 * (headline["value"] / headline["value"].shift(12) - 1)

df = (
    trimmed[["date", "period", "trimmed_mean_12m"]]
    .merge(skew[["date", "kelly_skewness_12m_avg"]], on="date")
    .merge(headline[["date", "headline_pce_12m"]], on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

df["gap_tm_minus_headline_12m"] = df["trimmed_mean_12m"] - df["headline_pce_12m"]
df["bias_tm_centered_36m"] = centered_trimmed_average(df["gap_tm_minus_headline_12m"])

latest_data_date = df.dropna(
    subset=["trimmed_mean_12m", "headline_pce_12m", "kelly_skewness_12m_avg"]
)["date"].max()
regression_end = latest_data_date - pd.DateOffset(months=FUTURE_MONTHS_NEEDED)

sample = df[
    df["date"].between(REGRESSION_START, regression_end)
    & df["bias_tm_centered_36m"].notna()
    & df["kelly_skewness_12m_avg"].notna()
].copy()

b, se, tstat, r2, n = ols_hac(sample["bias_tm_centered_36m"], sample["kelly_skewness_12m_avg"], HAC_LAGS)
alpha, beta = b

df["fitted_bias_pp"] = alpha + beta * df["kelly_skewness_12m_avg"]
df["bias_adjustment_pp"] = -df["fitted_bias_pp"]
df["trimmed_mean_12m_bias_adjusted"] = df["trimmed_mean_12m"] + df["bias_adjustment_pp"]

out_cols = [
    "date",
    "period",
    "headline_pce_12m",
    "trimmed_mean_12m",
    "kelly_skewness_12m_avg",
    "gap_tm_minus_headline_12m",
    "bias_tm_centered_36m",
    "fitted_bias_pp",
    "bias_adjustment_pp",
    "trimmed_mean_12m_bias_adjusted",
]
df[out_cols].to_csv(OUTPUT_CSV, index=False)

table1 = pd.DataFrame(
    {
        "Trimmed-Mean PCE Inflation": [
            f"{alpha:.2f}",
            f"({tstat[0]:.2f})",
            f"{beta:.2f}",
            f"({tstat[1]:.2f})",
            f"{r2:.2f}",
            f"{n:,}",
            f"{sample['date'].min():%b %Y}-{sample['date'].max():%b %Y}",
        ]
    },
    index=["alpha", "  t-stat", "beta", "  t-stat", "R2", "N", "Sample"],
)

latest = (
    df.dropna(subset=["trimmed_mean_12m_bias_adjusted"])
    .tail(6)[
        [
            "period",
            "headline_pce_12m",
            "trimmed_mean_12m",
            "kelly_skewness_12m_avg",
            "bias_adjustment_pp",
            "trimmed_mean_12m_bias_adjusted",
        ]
    ]
    .round(3)
)

print(f"Latest data date: {latest_data_date:%Y-%m}")
print(f"Regression sample: {sample['date'].min():%Y-%m} to {sample['date'].max():%Y-%m}")
print(f"Saved: {OUTPUT_CSV}")
display(table1)
display(latest)


Latest data date: 2026-04
Regression sample: 1986-06 to 2024-10
Saved: C:\Users\abour\OneDrive\Economic Research\Inflation\Trimmed Mean PCE\outputs\bias_adjusted_trimmed_mean_pce.csv


,Trimmed-Mean PCE Inflation
alpha,-0.27
t-stat,(-6.48)
beta,-2.27
t-stat,(-11.76)
R2,0.53
N,461
Sample,Jun 1986-Oct 2024


,period,headline_pce_12m,trimmed_mean_12m,kelly_skewness_12m_avg,bias_adjustment_pp,trimmed_mean_12m_bias_adjusted
550,2025-11,2.820,2.509,-0.022,0.221,2.730
551,2025-12,2.878,2.437,0.005,0.282,2.719
552,2026-01,2.858,2.411,0.013,0.299,2.710
553,2026-02,2.858,2.333,0.020,0.316,2.649
554,2026-03,3.525,2.356,0.074,0.438,2.795
555,2026-04,3.767,2.345,0.113,0.527,2.872
